# EMA Crossover Backtest — Exploratory Analysis

End-to-end walkthrough: data download → strategy → backtest → optimisation → visualisation.

In [1]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = 'notebook'

from src.data_loader import download_data, validate_data
from src.indicators import add_emas
from src.signals import generate_signals, get_trade_events
from src.backtester import run_backtest
from src.metrics import compute_all_metrics
from src.optimization import grid_search, best_params, build_heatmap_pivot
from src.visualization import (
    plot_price_with_signals, plot_equity_curve, plot_optimization_heatmap,
    plotly_price_chart, plotly_equity_chart, plotly_heatmap,
)

ModuleNotFoundError: No module named 'plotly'

## 1 · Download Data

In [ ]:
TICKER = 'SPY'
START  = '2018-01-01'
SHORT_EMA = 10
LONG_EMA  = 50

df_raw = download_data(TICKER, start=START)
validate_data(df_raw)
print(df_raw.shape)
df_raw.tail()

## 2 · Add Indicators & Signals

In [ ]:
df = add_emas(df_raw, SHORT_EMA, LONG_EMA)
df = generate_signals(df)
df = run_backtest(df)
df[['Close','EMA_short','EMA_long','Position','Signal','Strategy_Return','Equity_Curve']].tail(10)

## 3 · Performance Metrics

In [ ]:
metrics = compute_all_metrics(df)
for k, v in metrics.items():
    if 'Return' in k or 'Drawdown' in k or 'CAGR' in k or 'Volatility' in k:
        print(f"{k:<25} {v*100:+.2f} %")
    else:
        print(f"{k:<25} {v:.3f}")

## 4 · Static Matplotlib Charts

In [ ]:
plot_price_with_signals(df, ticker=TICKER)
plot_equity_curve(df, ticker=TICKER)

## 5 · Interactive Plotly Charts

In [ ]:
plotly_price_chart(df, ticker=TICKER, short=SHORT_EMA, long=LONG_EMA).show()

In [ ]:
plotly_equity_chart(df, ticker=TICKER).show()

## 6 · Parameter Optimisation (Grid Search)

In [ ]:
opt_df = grid_search(df_raw, short_range=range(5, 51, 5), long_range=range(20, 201, 10))
print('Best parameters:', best_params(opt_df))
opt_df.head(10)

In [ ]:
pivot = build_heatmap_pivot(opt_df, 'Sharpe')
plotly_heatmap(pivot, 'Sharpe').show()

In [ ]:
plot_optimization_heatmap(pivot, 'Sharpe')